# Fall 2024 Data Science Track: Week 2 - Data Cleaning Exercise

## Packages, Packages, Packages!

Import *all* the things here! You need the standard stuff: `pandas` and `numpy`.

If you got more stuff you want to use, add them here too. 🙂

In [1]:
# If needed, uncomment this once in your notebook environment:
# %pip install pandas numpy

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

# Support running from the exercise folder, week folder, or repository root.
week_folder = 'Week-02-Pandas-and-How-to-Make-a-Sick-Project'
candidates = [Path('../data'), Path('data'), Path(week_folder) / 'data']
data_dir = next((d for d in candidates if (d / 'food_coded.csv').is_file()), None)
if data_dir is None:
    raise FileNotFoundError('Run from the exercise folder, week folder, or repository root.')

## Introduction

With the packages out of the way, now you will be working with the following data sets:

* `food_coded.csv`: [Food choices](https://www.kaggle.com/datasets/borapajo/food-choices?select=food_coded.csv) from Kaggle
* `Ask A Manager Salary Survey 2021 (Responses) - Form Responses 1.tsv`: [Ask A Manager Salary Survey 2021 (Responses)](https://docs.google.com/spreadsheets/d/1IPS5dBSGtwYVbjsfbaMCYIWnOuRmJcbequohNxCyGVw/view?&gid=1625408792) as *Tab Separated Values (.tsv)* from Google Docs

Each one poses different challenges. But you’ll―of course―overcome them with what you learned in class! 😉

## Food Choices Data Set

### Load the Data

Load the Food choices data set into a new variable, `df_food`.

In [3]:
food_data_set_path = data_dir / 'food_coded.csv'
df_food = pd.read_csv(food_data_set_path)
food_original = df_food.copy(deep=True)

### Explore the Data

How much data did you just load?

In [4]:
df_food.shape

(125, 61)


In [5]:
print(f'{df_food.shape[0]:,} rows and {df_food.shape[1]} columns')
display(df_food.head())

125 rows and 61 columns
     GPA  Gender  ...  waffle_calories                    weight
0    2.4       2  ...             1315                       187
1  3.654       1  ...              900                       155
2    3.3       1  ...              900  I'm not answering this. 
3    3.2       1  ...             1315             Not sure, 240
4    3.5       1  ...              760                       190

[5 rows x 61 columns]


What are the columns and their types in this data set?

In [6]:
df_food.dtypes

GPA                     str
Gender                int64
breakfast             int64
calories_chicken      int64
calories_day        float64
                     ...   
type_sports             str
veggies_day           int64
vitamins              int64
waffle_calories       int64
weight                  str
Length: 61, dtype: object


In [7]:
pd.set_option('display.max_rows', 100)
df_food.dtypes.astype(str).rename('dtype').to_frame()

                                dtype
GPA                               str
Gender                          int64
breakfast                       int64
calories_chicken                int64
calories_day                  float64
calories_scone                float64
coffee                          int64
comfort_food                      str
comfort_food_reasons              str
comfort_food_reasons_coded    float64
cook                          float64
comfort_food_reasons_coded.1    int64
cuisine                       float64
diet_current                      str
diet_current_coded              int64
drink                         float64
eating_changes                    str
eating_changes_coded            int64
eating_changes_coded1           int64
eating_out                      int64
employment                    float64
ethnic_food                     int64
exercise                      float64
father_education              float64
father_profession                 str
fav_cuisine 

### Clean the Data

Perhaps we’d like to know more another day, but the team is really interested in just the relationship between calories (`calories_day`) and weight. …and maybe gender.

Can you remove the other columns? (Assign the result to a new variable, `df_food_col_subset`.)

In [8]:
food_columns = ['calories_day', 'weight', 'Gender']
df_food_col_subset = df_food.loc[:, food_columns].copy()
df_food_col_subset.head()

   calories_day                    weight  Gender
0           NaN                       187       2
1           3.0                       155       1
2           4.0  I'm not answering this.        1
3           3.0             Not sure, 240       1
4           2.0                       190       1


In [9]:
df_food_drop = df_food.drop(columns=df_food.columns.difference(food_columns))
pd.testing.assert_frame_equal(df_food_drop[food_columns], df_food_col_subset)

In [10]:
df_food_filter = df_food.filter(items=food_columns).copy()
pd.testing.assert_frame_equal(df_food_filter, df_food_col_subset)

What about `NaN`s? How many are there?

In [11]:
print('Missing values in the selected columns:')
print(df_food_col_subset.isna().sum())
print('Total:', df_food_col_subset.isna().sum().sum())

Missing values in the selected columns:
calories_day    19
weight           2
Gender           0
dtype: int64
Total: 21


In [12]:
df_food_col_subset.isna().sum().rename('missing_values').to_frame()

              missing_values
calories_day              19
weight                     2
Gender                     0


In [13]:
missing_summary = df_food_col_subset.isna() \
    .sum() \
    .rename('missing_values') \
    .to_frame()
missing_summary

              missing_values
calories_day              19
weight                     2
Gender                     0


We gotta remove those `NaN`s―the entire row.

In [14]:
df_food_col_subset.dropna(inplace=True)
df_food_col_subset.shape

(104, 3)


In [15]:
pd.testing.assert_frame_equal(df_food, food_original)
print('The original df_food is unchanged; the subset was copied.')

The original df_food is unchanged; the subset was copied.


But what about the weird non-numeric values in the column obviously meant for numeric data?

Notice the data type of that column from when you got the types of all the columns?

If only we could convert the column to a numeric type and drop the rows with invalid values. 🤔

In [16]:
df_food_col_subset['weight'] = pd.to_numeric(df_food_col_subset['weight'], errors='coerce')
df_food_col_subset.dropna(subset=['weight'], inplace=True)
assert not df_food_col_subset.isna().any().any()
df_food_col_subset.dtypes

calories_day    float64
weight          float64
Gender            int64
dtype: object


Now this data seems reasonably clean for our purposes! 😁

Let’s save it somewhere to be shipped off to another teammate. 💾

In [17]:
food_output_path = data_dir / 'food_cleaned.csv'
df_food_col_subset.to_csv(food_output_path, index=False)
print(f'Saved {len(df_food_col_subset)} rows to {food_output_path}')

Saved 101 rows to ..\data\food_cleaned.csv


In [18]:
print('\n'.join(food_output_path.read_text(encoding='utf-8').splitlines()[:6]))

calories_day,weight,Gender
3.0,155.0,1
2.0,190.0,1
3.0,190.0,1
3.0,180.0,2
3.0,137.0,1


## Ask a Manager Salary Survey 2021 (Responses) Data Set

### Load the Data

Load the Ask A Manager Salary Survey 2021 (Responses) data set into a new variable, `df_salary`.

In [19]:
salary_data_set_path = data_dir / 'Ask A Manager Salary Survey 2021 (Responses) - Form Responses 1.tsv'
df_salary = pd.read_csv(salary_data_set_path, sep='\t')

Was that hard? 🙃

### Explore

You know the drill.

How much data did you just load?

In [20]:
print(f'{df_salary.shape[0]:,} rows and {df_salary.shape[1]} columns')
df_salary.head()

28,062 rows and 18 columns
            Timestamp  ... What is your race? (Choose all that apply.)
0  4/27/2021 11:02:10  ...                                       White
1  4/27/2021 11:02:22  ...                                       White
2  4/27/2021 11:02:38  ...                                       White
3  4/27/2021 11:02:41  ...                                       White
4  4/27/2021 11:02:42  ...                                       White

[5 rows x 18 columns]


What are the columns and their types?

In [21]:
df_salary.dtypes.astype(str).rename('dtype').to_frame()

                                                      dtype
Timestamp                                               str
How old are you?                                        str
What industry do you work in?                           str
Job title                                               str
If your job title needs additional context, ple...      str
What is your annual salary? (You'll indicate th...      str
How much additional monetary compensation do yo...  float64
Please indicate the currency                            str
If "Other," please indicate the currency here:          str
If your income needs additional context, please...      str
What country do you work in?                            str
If you're in the U.S., what state do you work in?       str
What city do you work in?                               str
How many years of professional work experience ...      str
How many years of professional work experience ...      str
What is your highest level of education 

Oh… Ugh! Give these columns easier names to work with first. 🙄

In [22]:
salary_columns = ['timestamp', 'age', 'industry', 'title', 'title_context', 'salary', 'additional_compensation', 'currency', 'other_currency', 'salary_context', 'country', 'state', 'city', 'total_yoe', 'field_yoe', 'highest_education_completed', 'gender', 'race']
assert len(df_salary.columns) == len(salary_columns)
df_salary.rename(columns=dict(zip(df_salary.columns, salary_columns)), inplace=True)
df_salary.columns

Index(['timestamp', 'age', 'industry', 'title', 'title_context', 'salary',
       'additional_compensation', 'currency', 'other_currency',
       'salary_context', 'country', 'state', 'city', 'total_yoe', 'field_yoe',
       'highest_education_completed', 'gender', 'race'],
      dtype='str')


It’s a lot, and that should not have been easy. 😏

You’re going to have a gander at the computing/tech subset first because thats *your* industry. But first, what value corresponds to that `industry`?

In [23]:
df_salary['industry'].value_counts(dropna=False)

industry
Computing or Tech                               4699
Education (Higher Education)                    2464
Nonprofits                                      2419
Health care                                     1896
Government and Public Administration            1889
                                                ... 
Undergrad student                                  1
Concrete Construction                              1
I'm currently a student and don't have a job       1
Student                                            1
Wine & Spirits                                     1
Name: count, Length: 1220, dtype: int64


That value among the top 5 is what you’re looking for innit? Filter out all the rows not in that industry and save it into a new variable, `df_salary_tech`. 

In [24]:
df_salary_tech = df_salary.loc[df_salary['industry'].eq('Computing or Tech')].copy()

Do a sanity check by counting.

In [25]:
assert len(df_salary_tech) == df_salary['industry'].eq('Computing or Tech').sum()
print(f'{len(df_salary_tech):,} computing/tech responses')

4,699 computing/tech responses


We are very interested in salary figures. But how many dollars 💵 is a euro 💶 or a pound 💷? That sounds like a problem for another day. 🫠

For now, let’s just look at U.S. dollars (`'USD'`).

In [26]:
df_salary_tech.drop(index=df_salary_tech.index[~df_salary_tech['currency'].eq('USD')], inplace=True)
assert df_salary_tech['currency'].eq('USD').all()

What we really want know is how each U.S. state pays in tech. What value in `country` represents the United States of America?

In [27]:
country_counts = df_salary_tech['country'].value_counts()
country_counts

country
United States                1576
USA                          1222
US                            412
U.S.                          108
United States of America       90
United States                  68
Usa                            59
USA                            56
usa                            28
United states                  23
united states                  14
Us                             12
us                              9
U.S.A.                          7
United States of America        7
Israel                          5
Canada                          4
U.S.                            2
United State of America         2
Unite States                    2
Australia                       2
UnitedStates                    2
India                           2
U.S                             2
Usa                             2
United States Of America        2
Spain                           2
Brazil                          2
United Kingdom                  2
united

### Clean the Data

Well, we can’t get our answers with what we currently have, so you’ll have to make some changes.

Let’s not worry about anything below the first 5 values for now. Convert the top 5 to a single canonical value―say, `'US'`, which is nice and short.

In [28]:
# The top five observed values all refer to the United States.
top_five_us = country_counts.head(5).index
print(top_five_us.tolist())
df_salary_tech.loc[df_salary_tech['country'].isin(top_five_us), 'country'] = 'US'

['United States', 'USA', 'US', 'U.S.', 'United States of America']


Have a look at the count of each unique country again now.

In [29]:
df_salary_tech['country'].value_counts()

country
US                           3408
United States                  68
Usa                            59
USA                            56
usa                            28
United states                  23
united states                  14
Us                             12
us                              9
U.S.A.                          7
United States of America        7
Israel                          5
Canada                          4
U.S.                            2
United State of America         2
Unite States                    2
Australia                       2
UnitedStates                    2
India                           2
U.S                             2
Usa                             2
United States Of America        2
Spain                           2
Brazil                          2
United Kingdom                  2
united States                   2
New Zealand                     2
Poland                          2
France                          2
U.S.A 

Did you notice anything interesting?

In [30]:
# Whitespace, capitalization and punctuation explain many remaining variants.
# Match entire normalized names so unrelated countries are not relabeled.
def normalize_country(country):
    key = country.astype('string').str.lower().str.replace(r'[^a-z]', '', regex=True)
    is_us = key.str.fullmatch(r'us|usa|unitedstates?(?:ofamerica)?|america', na=False)
    return country.astype('string').str.strip().mask(is_us, 'US')

df_salary_tech['country'] = normalize_country(df_salary_tech['country'])

In [31]:
df_salary_tech['country'].value_counts(dropna=False)

country
US                      3716
Canada                     5
Israel                     5
Unite States               2
Australia                  2
India                      2
Spain                      2
Brazil                     2
United Kingdom             2
New Zealand                2
Poland                     2
France                     2
Nigeria                    2
Uniyed states              1
Puerto Rico                1
Cuba                       1
Danmark                    1
Italy                      1
International              1
United Stated              1
Remote (philippines)       1
Singapore                  1
Uruguay                    1
Mexico                     1
United Stateds             1
ISA                        1
singapore                  1
Pakistan                   1
Netherlands                1
China                      1
San Francisco              1
Romania                    1
Japan                      1
United Stares              1
Austra

It’s looking good so far. Let’s find out the minimum, mean, and maximum (in that order) salary by state, sorted by the mean in descending order.

In [32]:
# Demonstrate the expected problem without stopping the rest of the notebook.
try:
    display(df_salary_tech.loc[df_salary_tech['country'].eq('US')]
            .groupby('state')['salary'].agg(['min', 'mean', 'max'])
            .sort_values('mean', ascending=False))
except (TypeError, ValueError):
    print('Salary is text, including comma separators; convert it before averaging.')

Salary is text, including comma separators; convert it before averaging.


 Well, pooh! We forgot that `salary` isn’t numeric. Something wrong must be fixed. 🤔

In [33]:
def numeric_salary(salary):
    return pd.to_numeric(salary.astype('string').str.replace(',', '', regex=False)
                         .str.replace('$', '', regex=False).str.strip(), errors='coerce')

df_salary_tech['salary'] = numeric_salary(df_salary_tech['salary'])
df_salary_tech.dropna(subset=['salary'], inplace=True)

Let’s try that again.

In [34]:
def state_summary(frame):
    return (frame.loc[frame['country'].eq('US')]
            .groupby('state')['salary'].agg(['min', 'mean', 'max'])
            .sort_values('mean', ascending=False))

state_summary(df_salary_tech)

                                   min           mean      max
state                                                         
Michigan, Texas, Washington     340000       340000.0   340000
California, Oregon              200000       200000.0   200000
California, Colorado            176000       176000.0   176000
Georgia, Massachusetts          175000       175000.0   175000
Florida                          28800  157457.232143  2600000
Alabama, District of Columbia   156000       156000.0   156000
California                           0   155224.89313   875000
Washington                          72  151309.876106   950000
New York                         14000   148157.66954   590000
Nevada                           38000       141310.0   425000
New Jersey, New York            135000       137500.0   140000
Massachusetts                    37000  135213.537954  1650000
District of Columbia             10000  132318.036364   267500
New Mexico                       70000       132200.0  

That did the trick! Now let’s narrow this to data 2021 and 2022 just because (lel). *(Hint: that timestamp column may not be a temporal type right now.)*

In [35]:
df_salary_tech['timestamp'] = pd.to_datetime(
    df_salary_tech['timestamp'], format='%m/%d/%Y %H:%M:%S', errors='coerce')
# The prose asks for 2021-2022, while the code comment includes 2023.
# Show both explicitly to resolve the ambiguity.
df_salary_tech_2021_2022 = df_salary_tech.loc[
    df_salary_tech['timestamp'].dt.year.isin([2021, 2022])].copy()
display(state_summary(df_salary_tech_2021_2022))
df_salary_tech_2021_2023 = df_salary_tech.loc[
    df_salary_tech['timestamp'].dt.year.isin([2021, 2022, 2023])].copy()
display(state_summary(df_salary_tech_2021_2023))
print('Counts for Delaware and West Virginia (2021-2022):')
print(df_salary_tech_2021_2022.loc[df_salary_tech_2021_2022['country'].eq('US')]
      .groupby('state')['salary'].count().reindex(['Delaware', 'West Virginia'], fill_value=0))
print('Very small samples give unstable comparisons; one observation makes min, mean and max identical.')

                                   min           mean      max
state                                                         
Michigan, Texas, Washington     340000       340000.0   340000
California, Oregon              200000       200000.0   200000
California, Colorado            176000       176000.0   176000
Georgia, Massachusetts          175000       175000.0   175000
Alabama, District of Columbia   156000       156000.0   156000
California                           0  155355.206422   875000
Washington                          72  151458.278107   950000
New York                         14000  148397.317003   590000
Nevada                           38000       141310.0   425000
New Jersey, New York            135000       137500.0   140000
Massachusetts                    37000  135444.192691  1650000
District of Columbia             10000  132318.036364   267500
New Mexico                       70000       132200.0   300000
Utah, Vermont                   130000       130000.0  

## Bonus

Clearly, we do not have enough data to produce useful figures for the level of specificity you’ve now reached. What do you notice about Delaware and West Virginia?

Let’s back out a bit and return to `df_salary` (which was the loaded data with renamed columns but *sans* filtering).

### Bonus #0

Apply the same steps as before to `df_salary`, but do not filter for any specific industry. Do perform the other data cleaning stuff, and get to a point where you can generate the minimum, mean, and maximum by state.

In [36]:
# Start from the full survey, without restricting industry.
df_salary_all = df_salary.loc[df_salary['currency'].eq('USD')].copy()
df_salary_all['country'] = normalize_country(df_salary_all['country'])
df_salary_all['salary'] = numeric_salary(df_salary_all['salary'])
df_salary_all['timestamp'] = pd.to_datetime(
    df_salary_all['timestamp'], format='%m/%d/%Y %H:%M:%S', errors='coerce')
df_salary_all['state'] = df_salary_all['state'].astype('string').str.strip()
df_salary_all = df_salary_all.loc[
    df_salary_all['country'].eq('US') &
    df_salary_all['timestamp'].dt.year.isin([2021, 2022])].dropna(subset=['salary', 'state']).copy()
salary_state_summary = state_summary(df_salary_all)
salary_state_summary

                                                       min      mean     max
state                                                                       
Michigan, Texas, Washington                         340000  340000.0  340000
Indiana, Ohio                                       245000  245000.0  245000
Colorado, Nevada                                    190000  190000.0  190000
California, Montana                                 185000  185000.0  185000
California, Texas                                   185000  185000.0  185000
...                                                    ...       ...     ...
Delaware, Pennsylvania                               35000   35000.0   35000
District of Columbia, Washington                     35000   35000.0   35000
Alabama, California                                  29120   29120.0   29120
District of Columbia, Maryland, Pennsylvania, V...   27040   27040.0   27040
Maryland, New York                                   14000   14000.0   14000

### Bonus #1

This time, format the table output nicely (*$12,345.00*) without modifying the values in the `DataFrame`. That is, `df_salary` should be identical before versus after running your code.

(*Hint: if you run into an error about `jinja2` perhaps you need to `pip install` something.*)

In [37]:
# Format a separate display table; the source values remain numeric.
salary_before_formatting = df_salary.copy(deep=True)
display(salary_state_summary.apply(lambda column: column.map('${:,.2f}'.format)))
pd.testing.assert_frame_equal(df_salary, salary_before_formatting)

                                                            min  ...          max
state                                                            ...             
Michigan, Texas, Washington                         $340,000.00  ...  $340,000.00
Indiana, Ohio                                       $245,000.00  ...  $245,000.00
Colorado, Nevada                                    $190,000.00  ...  $190,000.00
California, Montana                                 $185,000.00  ...  $185,000.00
California, Texas                                   $185,000.00  ...  $185,000.00
...                                                         ...  ...          ...
Delaware, Pennsylvania                               $35,000.00  ...   $35,000.00
District of Columbia, Washington                     $35,000.00  ...   $35,000.00
Alabama, California                                  $29,120.00  ...   $29,120.00
District of Columbia, Maryland, Pennsylvania, V...   $27,040.00  ...   $27,040.00
Maryland, New Yo

### Bonus #2

Filter out the non-single-states (e.g., `'California, Colorado'`) in the most elegant way possible (i.e., *not* by blacklisting all the bad values).

In [38]:
# A positive list admits only actual individual states (plus DC).
states = set('Alabama|Alaska|Arizona|Arkansas|California|Colorado|Connecticut|Delaware|Florida|Georgia|Hawaii|Idaho|Illinois|Indiana|Iowa|Kansas|Kentucky|Louisiana|Maine|Maryland|Massachusetts|Michigan|Minnesota|Mississippi|Missouri|Montana|Nebraska|Nevada|New Hampshire|New Jersey|New Mexico|New York|North Carolina|North Dakota|Ohio|Oklahoma|Oregon|Pennsylvania|Rhode Island|South Carolina|South Dakota|Tennessee|Texas|Utah|Vermont|Virginia|Washington|West Virginia|Wisconsin|Wyoming|District of Columbia'.split('|'))
df_salary_single_state = df_salary_all.loc[df_salary_all['state'].isin(states)].copy()
state_summary(df_salary_single_state)

                        min           mean      max
state                                              
California                0  114590.211016   875000
Washington               72  107261.657604  1260000
District of Columbia     40  106643.829042  1334782
New York                 80  105326.773191  3000000
New Jersey            14850  101312.104061  5000044
Massachusetts           155   98905.320555  1650000
Virginia                 57   94762.011583  1300000
Connecticut               0   93766.438298  1900000
Illinois                  0   90340.004163  1100000
Colorado                 65   89744.338633   630000
Maryland                  0   89628.711986   353200
Oregon                    0   89002.336026   320000
Texas                    80   88834.596308  1200000
Georgia                   0   86720.132075   860000
Utah                  18720   86505.072816   576000
Delaware              35000   86398.148936   220000
Pennsylvania             55   83288.468017  1100000
New Mexico  

### Bonus #3

Show the quantiles instead of just minimum, mean, and maximum―say 0%, 5%, 25%, 50%, 75%, 95%, and 100%. Outliers may be deceiving.

Sort by whatever interests you―like say the *50th* percentile.

And throw in a count by state too. It would be interesting to know how many data points contribute to the figures for each state. (*Hint: your nice formatting from Bonus #1 might not work this time around.* 😜)

In [39]:
quantiles = [0, .05, .25, .50, .75, .95, 1]
grouped_salary = df_salary_single_state.groupby('state')['salary']
salary_quantiles = grouped_salary.quantile(quantiles).unstack()
salary_quantiles.columns = [f'{q:.0%}' for q in quantiles]
salary_quantiles.insert(0, 'count', grouped_salary.count())
salary_quantiles.sort_values('50%', ascending=False, inplace=True)
formatted_quantiles = salary_quantiles.copy()
for column in formatted_quantiles.columns.drop('count'):
    formatted_quantiles[column] = formatted_quantiles[column].map('${:,.2f}'.format)
display(formatted_quantiles)

                      count          0%  ...          95%           100%
state                                    ...                            
California             2578       $0.00  ...  $220,000.00    $875,000.00
Washington             1177      $72.00  ...  $200,600.00  $1,260,000.00
District of Columbia    971      $40.00  ...  $189,500.00  $1,334,782.00
New York               2156      $80.00  ...  $210,250.00  $3,000,000.00
Massachusetts          1513     $155.00  ...  $183,200.00  $1,650,000.00
Maryland                559       $0.00  ...  $165,000.00    $353,200.00
Connecticut             235       $0.00  ...  $163,000.00  $1,900,000.00
Virginia                777      $57.00  ...  $180,000.00  $1,300,000.00
Delaware                 47  $35,000.00  ...  $163,297.00    $220,000.00
Oregon                  619       $0.00  ...  $185,000.00    $320,000.00
Colorado                629      $65.00  ...  $165,300.00    $630,000.00
Illinois               1201       $0.00  ...  $170,